# ReAct + Calculator (Qwen3-4B) trên benchmark Số phức v2

**Benchmark:** 2070 câu v2, 23 dạng (đã bỏ D34), repo `slm-highschool-math-tutoring`, commit `3fe21c9`.

- Mỗi dạng chạy **đúng mã của notebook thử nghiệm cuối cùng đã đạt 90/90 trên v1** (prompt, Calculator, vòng lặp ReAct, tham số sinh, cách trích đáp án), lấy nguyên văn từ `data/pipeline_bundle/pipeline_units.json`, cũng là mã mà pipeline toàn hệ thống dùng. Mã nạp unit, tạo engine vLLM và chạy là các ô của notebook pipeline, giữ nguyên.
- Không có bộ phân dạng: mỗi câu được giải bằng prompt của **đúng dạng của nó** (để đo riêng phần giải).
- Tham số sinh như thử nghiệm gốc: `temperature=0.3, top_p=0.95, top_k=20, seed=42, presence_penalty=1.2`, tối đa 9216 token sinh.

**Cách chạy trên mỗi tài khoản (5 tài khoản):**
1. Settings: Accelerator = **GPU T4 x2**; **Internet = On**. Không cần Add Data.
2. Ở ô đầu tiên, đổi `SHARD_ID` thành số của tài khoản: 0, 1, 2, 3, 4.
3. Save Version, chọn **Save & Run All (Commit)**.
4. Khi chạy xong, tải `react_v2_shardKof5.zip` trong tab **Output**.

| Shard | Dạng |
|---|---|
| 0 | D50, D39, D48, D40, D04 |
| 1 | D46, D53, D32, D35, D44 |
| 2 | D52, D49, D45, D51, D28 |
| 3 | D36, D47, D42, D33 |
| 4 | D38, D37, D54, D14 |

In [ ]:
SHARD_ID = 0            # <-- DOI SO NAY cho moi tai khoan: 0, 1, 2, 3, 4
SHARDS = [["D50", "D39", "D48", "D40", "D04"], ["D46", "D53", "D32", "D35", "D44"], ["D52", "D49", "D45", "D51", "D28"], ["D36", "D47", "D42", "D33"], ["D38", "D37", "D54", "D14"]]

In [ ]:
VLLM_SPEC = 'vllm'      # neu van loi khoi dong engine, thu 'vllm==0.31.0' (ban moi nhat truoc ngay chay pipeline 2026-10-16)
# Same order as the pipeline notebook (torchao is removed before and after the installs; see its README).
!pip uninstall -y -q torchao
!pip install -q -U {VLLM_SPEC}
!pip uninstall -y -q torchcodec torchaudio
!pip uninstall -y -q torchao
import importlib.util, importlib.metadata as md, subprocess, sys, os, re
assert importlib.util.find_spec('torchao') is None, 'torchao is still installed'

# vLLM >= 0.3x imports `cutlass` (package nvidia-cutlass-dsl) when the engine warms up. The package exposes `cutlass` only
# through a .pth file run at interpreter start-up, which does not take effect in this already-running kernel or in an
# environment that already had an older copy. Make it importable for this kernel AND for the worker processes vLLM spawns.
def cutlass_ok():
    return subprocess.run([sys.executable, '-c', 'import cutlass'], capture_output=True, text=True)

r = cutlass_ok()
if r.returncode != 0:
    print('cutlass not importable:', r.stderr.strip().splitlines()[-1] if r.stderr.strip() else '')
    want = [x for x in (md.requires('vllm') or []) if x.lower().startswith('nvidia-cutlass-dsl')]
    spec = want[0].split(';')[0].strip() if want else 'nvidia-cutlass-dsl'
    print('installing', spec)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', spec])
    r = cutlass_ok()
if r.returncode != 0:
    import nvidia_cutlass_dsl
    p = os.path.join(list(nvidia_cutlass_dsl.__path__)[0], 'dsl_packages')
    os.environ['PYTHONPATH'] = p + os.pathsep + os.environ.get('PYTHONPATH', '')   # inherited by the vLLM workers
    sys.path.insert(0, p)
    r = cutlass_ok()
assert r.returncode == 0, 'cutlass still cannot be imported:\n' + r.stderr[-800:]
print('cutlass OK')
import transformers, torch, vllm
print('vllm', vllm.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__)

In [ ]:
import os, re, sys, json, glob, time, types, shutil, hashlib, subprocess, urllib.request
import pandas as pd

SHARD_ID = int(os.environ.get('SHARD_ID', SHARD_ID))
assert 0 <= SHARD_ID < len(SHARDS)
MY_TYPES = SHARDS[SHARD_ID]
REPO = 'https://raw.githubusercontent.com/manhdungcr7/slm-highschool-math-tutoring/3fe21c94a31cc62209e5004f688ac3fd04ddfe2c/'
FILES = {'pipeline_units.json': ('data/pipeline_bundle/pipeline_units.json', '1e7650bd01d044b96f25752856804a8b'),
         'benchmark_v2.csv': ('so_phuc/data/questions/benchmark_so_phuc_v2_23dang_2070cau.csv', '5ebfa1f6e4fcc2135d998b754d43b3bd')}
WORK_DIR = f'/kaggle/working/react_v2_shard{SHARD_ID}of{len(SHARDS)}'
RAW_DIR = os.path.join(WORK_DIR, 'raw')
os.makedirs(RAW_DIR, exist_ok=True)
for name, (path, want) in FILES.items():
    dst = os.path.join('/kaggle/working', name)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(REPO + path, dst)
    got = hashlib.md5(open(dst, 'rb').read()).hexdigest()
    assert got == want, f'{name}: md5 {got} != {want}'

PIPE = json.load(open('/kaggle/working/pipeline_units.json', encoding='utf-8'))
TYPE_BY_NO = {t['so_dang']: t for t in PIPE['types']}
UNITS = PIPE['units']
SP_TYPE = {t['ma_du_lieu']: t for t in PIPE['types'] if t['chu_de'] == 'Số phức'}
bench = pd.read_csv('/kaggle/working/benchmark_v2.csv', encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert len(bench) == 2070 and bench['ma_dang_bai_bao'].nunique() == 23 and 'D34' not in set(bench['ma_dang_bai_bao'])

In [ ]:
plan_rows = []
for r in bench[bench['ma_dang_bai_bao'].isin(MY_TYPES)].itertuples():
    t = SP_TYPE[r.ma_dang_du_lieu]                   # the unit of the question's OWN type (no router)
    assert t['cach_giai'] == 'ReAct+Calculator', (r.stt, t['ma_cu'])
    plan_rows.append(dict(STT=int(r.stt), ma_dang=r.ma_dang_bai_bao, ma_du_lieu=r.ma_dang_du_lieu, bien_the=int(r.bien_the),
                          so_dang=t['so_dang'], unit=t['unit'], de_bai_mcq=r.de_bai, dap_an_dung=r.dap_an_dung))
plan = pd.DataFrame(plan_rows)
assert len(plan) == 90 * len(MY_TYPES) and plan['STT'].is_unique
print(f'shard {SHARD_ID}/{len(SHARDS)}: {len(plan)} questions |', plan.groupby(['unit', 'ma_dang']).size().to_dict())

## Bước 2. Nạp mã của từng dạng, rồi nạp Qwen3-4B

Mã của mỗi notebook gốc chạy trong một module riêng (mỗi notebook có Calculator, vòng lặp và hằng số riêng nên không được trộn lẫn). Phần cấu hình và Calculator của các notebook Số phức tạo tiến trình con giới hạn thời gian (`fork`), nên phải chạy **trước** khi CUDA khởi tạo, giống thứ tự trong notebook gốc.

In [ ]:
MODS = {}


def run_code(mod, code):
    exec(compile(code, f'<{mod.__name__}>', 'exec'), mod.__dict__)


def new_module(unit):
    # A real module in sys.modules: functions sent to the Calculator's subprocess pool are pickled by module name.
    mod = types.ModuleType('unit_' + unit)
    sys.modules[mod.__name__] = mod
    return mod


needed = sorted(plan['unit'].unique())
for unit in needed:
    u = UNITS[unit]
    mod = new_module(unit)
    run_code(mod, u['setup'])
    if u['kind'] in ('ml_group', 'ml_single', 'sp_single', 'ml_zeroshot'):
        run_code(mod, u['prompt'])          # only string constants
    MODS[unit] = mod
    print(f"{unit:10s} {u['kind']:12s} <- {u['source']}")


def md5(s):
    return hashlib.md5(s.encode('utf-8')).hexdigest()


# The prompts now in memory are the ones of the experiments (same md5 as recorded from the original notebooks).
for unit in needed:
    u, mod = UNITS[unit], MODS[unit]
    if u['kind'] == 'ml_group':
        for ma in sorted(set(plan.loc[plan.unit == unit, 'ma_du_lieu'])):
            assert md5(mod.TYPE_PROMPTS[ma]['template']) == u['prompt_md5'][ma], (unit, ma)
    elif u['kind'] in ('ml_single', 'sp_single'):
        assert md5(mod.PROMPT_TEMPLATE) == list(u['prompt_md5'].values())[0], unit
    elif u['kind'] == 'ml_zeroshot':
        assert md5(mod.PROMPT_TEMPLATE) == u['prompt_md5'], unit
print('Prompts checked.')

In [ ]:
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

MODEL, SEED = 'Qwen/Qwen3-4B', 42
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)


def routed(unit, ma=None):
    sub = plan[plan.unit == unit]
    return sub if ma is None else sub[sub.ma_du_lieu == ma]


def ml_model_len(mod, pairs):
    """Cell 4 of the Mũ-Logarit ReAct notebooks: the longest prompt plus the whole generation budget."""
    prompt_tokens = 0
    for template, sub in pairs:
        longest = max(sub['de_bai_mcq'], key=len)
        prompt_tokens = max(prompt_tokens, len(tok(template.replace('{de_bai}', longest)).input_ids))
    return ((prompt_tokens + mod.MAX_NEW_TOKENS + 1500) // 1024 + 1) * 1024


for unit in needed:
    u, mod = UNITS[unit], MODS[unit]
    if u['kind'] == 'ml_group':
        mod.MAX_MODEL_LEN = ml_model_len(mod, [(mod.TYPE_PROMPTS[ma]['template'], routed(unit, ma))
                                               for ma in sorted(set(routed(unit).ma_du_lieu))])
    elif u['kind'] == 'ml_single':
        mod.MAX_MODEL_LEN = ml_model_len(mod, [(mod.PROMPT_TEMPLATE, routed(unit))])
    print(f'{unit:10s} MAX_MODEL_LEN={mod.MAX_MODEL_LEN}')
ENGINE_LEN = max(MODS[u].MAX_MODEL_LEN for u in needed)

N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'GPUs: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}, engine max_model_len={ENGINE_LEN}')
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=ENGINE_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
for mod in MODS.values():
    mod.tok, mod.llm = tok, llm
print('Ready.')

## Bước 3. Giải

Mỗi nhóm câu chạy bằng đúng mã của notebook gốc. Kết quả từng nhóm được lưu ngay vào `raw/` nên vẫn xem được nếu phiên bị dừng giữa chừng.

In [ ]:
OUTPUTS = {}          # STT -> answer and solution


def ml_records(sub):
    return [{'STT': int(r.STT), 'ma_dang': r.ma_du_lieu, 'dap_an_letter': r.dap_an_dung, 'de_bai_mcq': r.de_bai_mcq}
            for r in sub.itertuples()]


def sp_records(sub):
    return [{'STT': int(r.STT), 'loai_so': '', 'de_bai_mcq': r.de_bai_mcq, 'dap_an_letter': r.dap_an_dung}
            for r in sub.itertuples()]


def keep(df, unit, answer, text, tokens='token', calls=None, stop=None):
    df.to_csv(os.path.join(RAW_DIR, f'{unit}.csv' if 'ma' not in df.attrs else f"{unit}_{df.attrs['ma']}.csv"),
              index=False, encoding='utf-8-sig')
    for _, r in df.iterrows():
        OUTPUTS[int(r['STT'])] = dict(model_chon=r[answer] if isinstance(r[answer], str) else '',
                                      token=int(r[tokens]) if tokens else None,
                                      so_luot_goi_tool=int(r[calls]) if calls else 0,
                                      ly_do_dung=r[stop] if stop else '', loi_giai=r[text])


def move(path):
    df = pd.read_csv(path, encoding='utf-8-sig', keep_default_na=False)
    os.remove(path)
    return df


T0 = time.time()
for unit in needed:
    u, mod, sub = UNITS[unit], MODS[unit], routed(unit)
    t1 = time.time()
    print('#' * 72)
    print(f"{unit}: {len(sub)} questions, {u['kind']}  ({(t1 - T0) / 60:.1f} min so far)")
    print('#' * 72)
    if u['kind'] == 'ml_zeroshot':
        # One call for every Mũ-Logarit zero-shot question: same prompt and sampling for all these dạng.
        mod.TYPES = ['pipeline_ml']
        mod.ALL_RECORDS = {'pipeline_ml': ml_records(sub)}
        run_code(mod, u['sampling'])
        run_code(mod, u['loop'])
        df = move('/kaggle/working/pipeline_ml_zeroshot_full90_Qwen3-4B.csv')
        df['loi_giai'] = df['think'] + '</think>' + df['after']
        keep(df, unit, 'model_chon', 'loi_giai')
    elif u['kind'] == 'sp_zeroshot':
        mod.records = [dict(stt=int(r.STT), stt_goc=None, van_de=None, ma_dang=r.ma_du_lieu, muc_do=None, ten_dang=None,
                            nguon='Thực tế', loai_so=None, de_bai=r.de_bai_mcq, correct=r.dap_an_dung)
                       for r in sub.itertuples()]
        run_code(mod, u['prompt'])
        mod.tok = tok
        run_code(mod, u['gen'])
        run_code(mod, u['extract'])
        df = mod.res.rename(columns={'stt': 'STT'})
        df['token'] = [len(o.outputs[0].token_ids) for o in mod.outputs]
        keep(df, unit, 'qwen_chon', 'loi_giai_qwen')
    elif u['kind'] == 'ml_group':
        mod.ALL_RECORDS = {ma: ml_records(routed(unit, ma)) for ma in sorted(set(sub.ma_du_lieu))}
        run_code(mod, u['loop'])
        for ma in mod.ALL_RECORDS:
            mod.run_type(ma)
            df = move(f'/kaggle/working/{ma.lower()}_react_calculator_full90.csv')
            df.attrs['ma'] = ma
            keep(df, unit, 'model_chon', 'full_text', calls='tool_calls', stop='stop_reason')
    elif u['kind'] == 'ml_single':
        mod.records = ml_records(sub)
        mod.OUT_PATH = os.path.join(RAW_DIR, f'{unit}_out.csv')
        run_code(mod, u['loop'])
        keep(move(mod.OUT_PATH), unit, 'model_chon', 'full_text', calls='tool_calls', stop='stop_reason')
    elif u['kind'] == 'sp_single':
        mod.records = sp_records(sub)
        mod.OUT_PATH = os.path.join(RAW_DIR, f'{unit}_out.csv')
        run_code(mod, u['loop'])
        keep(move(mod.OUT_PATH), unit, 'model_chon', 'full_text', calls='so_luot_goi_tool', stop='ly_do_dung')
    elif u['kind'] == 'sp_group':
        unit_dir = os.path.join(WORK_DIR, 'data_' + unit)
        os.makedirs(unit_dir, exist_ok=True)
        mod.DATA_DIR = mod.OUT_DIR = unit_dir
        run_code(mod, u['loop'])                      # defines chay_mot_dang(...)
        for ma in sorted(set(sub.ma_du_lieu)):
            json.dump(sp_records(routed(unit, ma)), open(os.path.join(unit_dir, u['data_file'][ma]), 'w', encoding='utf-8'),
                      ensure_ascii=False)
            run_code(mod, u['type_cells'][ma])        # the cell of that dạng: its prompt, then chay_mot_dang(...)
            df = move(os.path.join(unit_dir, u['out_file'][ma]))
            df.attrs['ma'] = ma
            keep(df, unit, 'model_chon', 'full_text', calls='so_luot_goi_tool', stop='ly_do_dung')
    print(f'{unit} done in {(time.time() - t1) / 60:.1f} min; {len(OUTPUTS)}/{len(plan)} questions answered so far.')
print(f'All done in {(time.time() - T0) / 60:.1f} min.')
assert sorted(OUTPUTS) == sorted(plan.STT), 'some questions have no output'

In [ ]:
res = plan.drop(columns=['de_bai_mcq']).copy()
for k in ('model_chon', 'token', 'so_luot_goi_tool', 'ly_do_dung', 'loi_giai'):
    res[k] = res['STT'].map(lambda s: OUTPUTS[s][k])
res['ket_qua'] = ['Không xác định' if c == '' else ('Đúng' if c == d else 'Sai')
                  for c, d in zip(res['model_chon'], res['dap_an_dung'])]
res = res.sort_values('STT')
res.to_csv(os.path.join(WORK_DIR, f'react_v2_shard{SHARD_ID}of{len(SHARDS)}.csv'), index=False, encoding='utf-8-sig')
summ = dict(shard=SHARD_ID, n_shards=len(SHARDS), types=MY_TYPES, n=len(res), minutes=round((time.time() - T0) / 60, 1),
            dung=int((res['ket_qua'] == 'Đúng').sum()), sai=int((res['ket_qua'] == 'Sai').sum()),
            khong_xac_dinh=int((res['ket_qua'] == 'Không xác định').sum()),
            theo_dang={m: f"{int((g['ket_qua'] == 'Đúng').sum())}/{len(g)}" for m, g in res.groupby('ma_dang')})
json.dump(summ, open(os.path.join(WORK_DIR, f'react_v2_summary_shard{SHARD_ID}of{len(SHARDS)}.json'), 'w', encoding='utf-8'),
          ensure_ascii=False, indent=1)
print(json.dumps(summ, ensure_ascii=False, indent=1))
zip_path = shutil.make_archive(f'/kaggle/working/react_v2_shard{SHARD_ID}of{len(SHARDS)}', 'zip', WORK_DIR)
print('Saved:', zip_path)